<a href="https://colab.research.google.com/github/juli0AND/Evaluacion-comparativa-de-YOLOv5-y-YOLOv8/blob/main/YOLOV8m.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# INSTALAR LIBRERÍAS
# ==========================================================
!pip install ultralytics roboflow thop pandas -q

# ==========================================================
# IMPORTAR LIBRERÍAS
# ==========================================================
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import Image, display
from google.colab import files

import torch
import time
import numpy as np
import pandas as pd
import os

from thop import profile

# ==========================================================
# DESCARGAR DATASET DESDE ROBOFLOW
# ==========================================================
rf = Roboflow(api_key="io4o0d2qhkzDHpqNE3Tb")

project = rf.workspace("deteccion-residuos") \
            .project("new-classification-yolov8_medium-josue")

version = project.version(1)

dataset = version.download("yolov8")

# ==========================================================
# DESACTIVAR WANDB
# ==========================================================
os.environ["WANDB_MODE"] = "disabled"

# ==========================================================
# ENTRENAR MODELO YOLOv8
# ==========================================================
# Determinar el dispositivo a usar (GPU si está disponible, de lo contrario CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO("yolov8m.pt")
model.to(device) # Mover el modelo al dispositivo correcto
# Cambiar por:
# yolov8s.pt
# yolov8m.pt
# según la versión a evaluar

model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=50,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    name="yolov8_residuos"
)

# ==========================================================
# VALIDACIÓN
# ==========================================================
metrics = model.val()

# ==========================================================
# OBTENER MÉTRICAS AUTOMÁTICAMENTE
# ==========================================================
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

print("\n======== MÉTRICAS ========")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"mAP@50: {map50:.4f}")
print(f"mAP@50-95: {map5095:.4f}")

# ==========================================================
# MOSTRAR GRÁFICAS
# ==========================================================
results_path = "runs/detect/yolov8_residuos/results.png"
confusion_path = "runs/detect/yolov8_residuos/confusion_matrix.png"

display(Image(results_path))
display(Image(confusion_path))

# ==========================================================
# CARGAR MODELO ENTRENADO
# ==========================================================
WEIGHTS = "runs/detect/yolov8_residuos/weights/best.pt"

model = YOLO(WEIGHTS)
model.to(device) # Mover el modelo al dispositivo correcto

# ==========================================================
# CONFIGURAR DISPOSITIVO
# ==========================================================
# La variable 'device' ya está definida arriba

# ==========================================================
# PARÁMETROS DEL MODELO
# ==========================================================
params = sum(p.numel() for p in model.model.parameters())
params_m = params / 1e6

# ==========================================================
# FLOPs
# ==========================================================
dummy = torch.randn(1, 3, 640, 640).to(device)

flops, _ = profile(
    model.model,
    inputs=(dummy,),
    verbose=False
)

flops_g = flops / 1e9

# ==========================================================
# TAMAÑO DEL MODELO
# ==========================================================
size_mb = os.path.getsize(WEIGHTS) / (1024 * 1024)

# ==========================================================
# WARM-UP
# ==========================================================
for _ in range(20):
    with torch.no_grad():
        _ = model.predict(dummy)

# ==========================================================
# MEDIR TIEMPO DE INFERENCIA
# ==========================================================
times = []

for _ in range(100):

    start = time.perf_counter()

    with torch.no_grad():
        _ = model.predict(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    times.append((end - start) * 1000)

avg_time = np.mean(times)
std_time = np.std(times)

fps = 1000 / avg_time

# ==========================================================
# TABLA RESUMEN
# ==========================================================
MODEL_NAME = "YOLOv8m_3"

df = pd.DataFrame({
    "Modelo":[MODEL_NAME],
    "Precision":[round(precision,4)],
    "Recall":[round(recall,4)],
    "mAP50":[round(map50,4)],
    "mAP50_95":[round(map5095,4)],
    "Tiempo_ms":[round(avg_time,2)],
    "FPS":[round(fps,2)],
    "Parametros_M":[round(params_m,2)],
    "FLOPs_GFLOPs":[round(flops_g,2)],
    "Tamano_MB":[round(size_mb,2)]
})

print("\n======== TABLA FINAL ========")
print(df)

# ==========================================================
# EXPORTAR CSV
# ==========================================================
csv_name = f"{MODEL_NAME}_resumen.csv"

df.to_csv(csv_name, index=False)

# ==========================================================
# EXPORTAR IMÁGENES EN ALTA RESOLUCIÓN (300 DPI)
# ==========================================================
from PIL import Image as PILImage

# RESULTS.PNG
img = PILImage.open(results_path)
img.save(
    "results_yolov8m_3_300dpi.png",
    dpi=(300,300)
)

# CONFUSION MATRIX
img2 = PILImage.open(confusion_path)
img2.save(
    "confusion_matrix_yolov8m_3_300dpi.png",
    dpi=(300,300)
)

# ==========================================================
# DESCARGAR ARCHIVOS IMPORTANTES
# ==========================================================
files.download(csv_name)

files.download("results_yolov8m_3_300dpi.png")

files.download("confusion_matrix_yolov8m_3_300dpi.png")

files.download(WEIGHTS)

files.download(
    "runs/detect/yolov8_residuos/results.csv"
)